# ANN index (HNSW)

Run after `python -m similarity.embeddings` and `python -m similarity.ann build`.
Needs faiss-cpu (requirements-dev.txt).

In [ ]:
# Work from the repo root, so `similarity` imports and the data/ paths resolve when opened from notebooks/
import os, sys
if os.path.basename(os.getcwd()) == "notebooks":
    os.chdir("..")
sys.path.insert(0, os.getcwd())

In [ ]:
import time
import faiss
import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import evals.ann_benchmark as bench
import similarity.ann as ann
import similarity.search as s
from similarity.preprocessing import load_products

products = load_products()
f = s.load_features()
index = ann.product_index(f)
vectors = index.reconstruct_n(0, index.ntotal)
index.ntotal, index.d, index.hnsw.max_level

## the layers

In [ ]:
levels = faiss.vector_to_array(index.hnsw.levels) - 1   # top layer each vector reaches
pd.Series(levels).value_counts().sort_index().rename("vectors")

## the product vector

In [ ]:
sizes = {"title": 384, "category": f.category.max() + 1 + f.browsenode.max() + 1, "gender": f.gender.max() + 1,
         "brand": ann.BRAND_SLOTS, "price": ann.price_bands(np.array([500.0])).shape[1]}
sizes, sum(sizes.values())

In [ ]:
bands = ann.price_bands(np.array([250.0, 500.0, 1000.0, 2000.0, np.nan]))
pd.DataFrame(bands @ bands.T, index=["250", "500", "1000", "2000", "missing"], columns=["250", "500", "1000", "2000", "missing"]).round(2)

In [ ]:
row = f.row_of["2304753b72be21b48128f7d2ae482c9f"]   # Q14 frock
print(products.loc[row, "name"])
score = s.combine(s.attribute_scores(f, row), s.WEIGHTS)
title = s.title_embeddings(f).vectors
exact_top = s.rank(f, row)[0][:10]

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
for ax, (name, v) in zip(axes, [("title only (MiniLM)", title), ("product vector", vectors)]):
    cosine = v @ v[row]
    cut = np.sort(cosine)[-ann.CANDIDATES]
    ax.scatter(cosine, score, s=2, alpha=0.3)
    ax.scatter(cosine[exact_top], score[exact_top], s=20, color="red", label="exact top 10")
    ax.axvline(cut, color="black", linestyle="--", label=f"shortlist of {ann.CANDIDATES}")
    ax.set_title(name)
    ax.set_xlabel("cosine to the query")
    ax.legend()
axes[0].set_ylabel("API score")
plt.tight_layout()

## step 1: HNSW vs exact, on the vectors alone

In [ ]:
rows = bench.sample_rows(f)
step1 = bench.measure_vectors(index, vectors, vectors[rows])
step1.round(3)

## which vector to shortlist with (exact shortlists, no HNSW yet)

In [ ]:
choice = {}
for name, v in [("title only", title), ("product vector", vectors)]:
    choice[name] = bench.measure_rerank(f, rows, lambda r, n, v=v: bench.top_by_vector(v, r, n), candidate_values=(250, 500, 1000))
pd.concat(choice).round(3)

## both steps: HNSW shortlist + re-rank

In [ ]:
both = bench.measure_rerank(f, rows, lambda r, n: ann.shortlist(index, vectors[r], n))
both.round(3)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ax.plot(both["ms per query"][1:], both["recall@10"][1:], marker="o", label="shortlist + re-rank")
for _, r in both[1:].iterrows():
    ax.annotate(r["search"].split()[1], (r["ms per query"], r["recall@10"]), textcoords="offset points", xytext=(4, -10))
ax.scatter(both["ms per query"][0], 1.0, color="black", label="exact")
ax.set_xlabel("ms per query")
ax.set_ylabel("recall@10")
ax.legend()
plt.tight_layout()

## one product

In [ ]:
def compare(uniq_id, k=10):
    row = f.row_of[uniq_id]
    print(products.loc[row, "name"])
    exact, exact_score = s.rank(f, row)
    found, found_score = ann.ann_rank(f, row)
    return pd.DataFrame({
        "exact": products.loc[exact[:k], "name"].str[:45].tolist(), "score": exact_score[:k].round(3),
        "ann": products.loc[found[:k], "name"].str[:45].tolist(), "ann score": found_score[:k].round(3),
    }, index=range(1, k + 1))

compare("2304753b72be21b48128f7d2ae482c9f")   # Q14 frock

In [ ]:
compare("0f8a3a26f3e563621eda640aa3ecd17e")   # Q15 kurta

## scale test

200,000 made-up vectors here (about 3 minutes). `python -m evals.ann_benchmark scale` runs the full 1,000,000-vector test.

In [ ]:
result = bench.scale_test(f, size=200_000)
result["vectors"].round(3)

In [ ]:
{k: round(v, 1) for k, v in result.items() if k in ('build_s', 'shortlist_ms', 'exact_full_ms')}

## evaluation

`python -m evals.run_eval --alternative tfidf_ann` writes the workbook.

In [ ]:
pd.read_excel("evals/eval_results.xlsx", sheet_name="Methods")